In [1]:
!pip install kaggle opencv-python tensorflow matplotlib scikit-learn

In [2]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"mrvillan09","key":"2de0babfec088e8e81201172304df6a9"}'}

In [3]:
import os

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [4]:
!kaggle datasets download -d vasukipatel/face-recognition-dataset
!unzip face-recognition-dataset.zip

Streaming output truncated to the last 5000 lines.
  inflating: Faces/Faces/Alexandra Daddario_76.jpg  
  inflating: Faces/Faces/Alexandra Daddario_77.jpg  
  inflating: Faces/Faces/Alexandra Daddario_78.jpg  
  inflating: Faces/Faces/Alexandra Daddario_79.jpg  
  inflating: Faces/Faces/Alexandra Daddario_8.jpg  
  inflating: Faces/Faces/Alexandra Daddario_80.jpg  
  inflating: Faces/Faces/Alexandra Daddario_81.jpg  
  inflating: Faces/Faces/Alexandra Daddario_82.jpg  
  inflating: Faces/Faces/Alexandra Daddario_83.jpg  
  inflating: Faces/Faces/Alexandra Daddario_84.jpg  
  inflating: Faces/Faces/Alexandra Daddario_85.jpg  
  inflating: Faces/Faces/Alexandra Daddario_86.jpg  
  inflating: Faces/Faces/Alexandra Daddario_87.jpg  
  inflating: Faces/Faces/Alexandra Daddario_88.jpg  
  inflating: Faces/Faces/Alexandra Daddario_89.jpg  
  inflating: Faces/Faces/Alexandra Daddario_9.jpg  
  inflating: Faces/Faces/Alexandra Daddario_90.jpg  
  inflating: Faces/Faces/Alexandra Daddario_91.jpg

In [6]:
import os
print(os.listdir())

['.config', 'Faces', 'face-recognition-dataset.zip', 'Original Images', 'Dataset.csv', 'kaggle.json', 'sample_data']


In [7]:
dataset_path = "Faces"

In [8]:
import os
import cv2
import numpy as np

dataset_path = "Faces"

data = []
labels = []
label_dict = {}
label_count = 0

for person in os.listdir(dataset_path):
    person_path = os.path.join(dataset_path, person)

    if not os.path.isdir(person_path):
        continue

    label_dict[label_count] = person

    for img in os.listdir(person_path):
        img_path = os.path.join(person_path, img)

        image = cv2.imread(img_path)
        if image is None:
            continue

        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        gray = cv2.resize(gray, (100,100))

        data.append(gray)
        labels.append(label_count)

    label_count += 1

data = np.array(data).reshape(-1,100,100,1) / 255.0
labels = np.array(labels)

print("Total Images:", len(data))
print("Total Classes:", len(label_dict))

Total Images: 2562
Total Classes: 1


In [9]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

X_train, X_test, y_train_raw, y_test_raw = train_test_split(
    data, labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

y_train = to_categorical(y_train_raw)
y_test = to_categorical(y_test_raw)

In [10]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization

model = Sequential([

    Conv2D(32, (3,3), activation='relu', input_shape=(100,100,1)),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),

    Dense(len(label_dict), activation='softmax')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [11]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [12]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rotation_range=15,
    zoom_range=0.1,
    horizontal_flip=True
)

datagen.fit(X_train)

In [13]:
history = model.fit(
    datagen.flow(X_train, y_train, batch_size=32),
    epochs=20,
    validation_data=(X_test, y_test)
)

Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
/usr/local/lib/python3.12/dist-packages/keras/src/ops/nn.py:944: UserWarning: You are using a softmax over axis -1 of a tensor of shape (None, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/losses/losses.py:33: SyntaxWarning: In loss categorical_crossentropy, expected y_pred.shape to be (batch_size, num_classes) with num_classes > 1. Received: y_pred.shape=(None, 1). Consider using 'binary_crossentropy' if you only have 2 classes.
  return self.f

65/65 ━━━━━━━━━━━━━━━━━━━━ 43s 613ms/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 2/20
65/65 ━━━━━━━━━━━━━━━━━━━━ 40s 613ms/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 3/20
65/65 ━━━━━━━━━━━━━━━━━━━━ 40s 594ms/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 4/20
65/65 ━━━━━━━━━━━━━━━━━━━━ 43s 670ms/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 5/20
65/65 ━━━━━━━━━━━━━━━━━━━━ 39s 604ms/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 6/20
65/65 ━━━━━━━━━━━━━━━━━━━━ 41s 635ms/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 7/20
65/65 ━━━━━━━━━━━━━━━━━━━━ 41s 625ms/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 8/20
65/65 ━━━━━━━━━━━━━━━━━━━━ 41s 630ms/step

In [14]:
loss, accuracy = model.evaluate(X_test, y_test)
print("Final Test Accuracy:", accuracy)

17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 112ms/step - accuracy: 1.0000 - loss: 0.0000e+00
Final Test Accuracy: 1.0
